<a href="https://colab.research.google.com/github/boruizhang/representations/blob/main/05_vibe_coding_tiny_english_splitter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Design your own prompt to your LLM, Let it write a small Python program that tries to split English words into prefixes and suffixes, or put morphemes together. Below is a tiny english splitter with two prompts.
---

### You may **not** use:
- Any corpora, APIs, or web scraping
- Any NLP libraries (e.g., NLTK, spaCy, etc.)
- Any pretrained models or embeddings

In [ ]:
# Example Word list
WORDS = [
    "unhappy", "redo", "replaying", "played", "happily",
    "pretest", "untied", "reheated", "unlikely", "runner",

    "undoable",
    "unlockable",
    "unresting",
    "reenter",
    "rewrite",

    "unreliable",
    "reusable",
    "preexisting",
    "disagreement",
    "modernization",
]


In [ ]:
PREFIXES = ["un", "re", "pre"]
SUFFIXES = ["ing", "ed", "ly", "er"]

def split_word(word):
    prefix = ""
    suffix = ""

    for p in PREFIXES:
        if word.startswith(p):
            prefix = p
            word = word[len(p):]
            break

    for s in SUFFIXES:
        if word.endswith(s):
            suffix = s
            stem = word[:-len(s)]
            return prefix, stem, suffix

    return prefix, word, suffix  # no suffix found

for w in WORDS:
    p, stem, s = split_word(w)
    print(f"{w:12} -> {p + '- ' if p else ''}{stem}{' -' + s if s else ''}")

My prompt: words like "undoable" can be split two ways. Make the function offer two all possible ways of splitting.

In [ ]:
PREFIXES = ["un", "re", "pre", "dis"]
SUFFIXES = ["able", "ing", "ed", "ly", "er", "ment", "ation", "ization"]

def all_splits(word):
    analyses = []

    # analysis 1: prefix-first, then suffix
    for p in [""] + PREFIXES:
        if p and not word.startswith(p):
            continue
        core1 = word[len(p):] if p else word

        for s in [""] + SUFFIXES:
            if s and not core1.endswith(s):
                continue
            stem1 = core1[:-len(s)] if s else core1
            if stem1:  # don't allow empty stem
                analyses.append((p, stem1, s))

    # analysis 2: suffix-first, then prefix (can yield different results)
    for s in [""] + SUFFIXES:
        if s and not word.endswith(s):
            continue
        core2 = word[:-len(s)] if s else word

        for p in [""] + PREFIXES:
            if p and not core2.startswith(p):
                continue
            stem2 = core2[len(p):] if p else core2
            if stem2:
                analyses.append((p, stem2, s))

    # dedupe
    unique = []
    seen = set()
    for a in analyses:
        if a not in seen:
            seen.add(a)
            unique.append(a)
    return unique

def pretty(p, stem, s):
    parts = []
    if p: parts.append(p + "-")
    parts.append(stem)
    if s: parts.append("-" + s)
    return "".join(parts)

for w in WORDS:
    analyses = all_splits(w)

    # only show "interesting" ones: those with at least one affix
    analyses = [a for a in analyses if a[0] or a[2]]

    print(f"\n{w}")
    for p, stem, s in analyses[:6]:  # cap output so it doesn't explode
        print("  ", pretty(p, stem, s))


unhappy
   un-happy

redo
   re-do

replaying
   replay-ing
   re-playing
   re-play-ing

played
   play-ed

happily
   happi-ly

pretest
   pre-test

untied
   unti-ed
   un-tied
   un-ti-ed

reheated
   reheat-ed
   re-heated
   re-heat-ed

unlikely
   unlike-ly
   un-likely
   un-like-ly

runner
   runn-er

undoable
   undo-able
   un-doable
   un-do-able

unlockable
   unlock-able
   un-lockable
   un-lock-able

unresting
   unrest-ing
   un-resting
   un-rest-ing

reenter
   reent-er
   re-enter
   re-ent-er

rewrite
   re-write

unreliable
   unreli-able
   un-reliable
   un-reli-able

reusable
   reus-able
   re-usable
   re-us-able

preexisting
   preexist-ing
   pre-existing
   pre-exist-ing

disagreement
   disagree-ment
   dis-agreement
   dis-agree-ment

modernization
   moderniz-ation
   modern-ization
